In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

HERE = Path.cwd()
REPO_ROOT = HERE.parents[3]

PCT_CSV = HERE / "first_second_win_pct_all_seeds.csv"
SPEARMAN_PER_LEAGUE = HERE / "spearman_first_second_by_league.csv"

FIG_DIR = HERE / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LEAGUE_NAME_MAP = {
    "bundesliga": "Bundesliga",
    "premier_league": "Premier League",
    "serie_a": "Serie A",
    "la_liga": "La Liga",
}

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
pct_df = pd.read_csv(PCT_CSV) # per league-season-team rows
sp_league = pd.read_csv(SPEARMAN_PER_LEAGUE) # per-league row

assert {"league", "season", "team", "first_half_win_pct_avg", "second_half_win_pct_avg"}.issubset(pct_df.columns)

pct_df["league"] = pct_df["league"].str.lower()
sp_league["league"] = sp_league["league"].str.lower()

In [ ]:
def plot_first_vs_second_avg_for_league(league_key: str):
    """
    Scatter: first_half_win_pct_avg vs second_half_win_pct_avg
    Points = teams across all seasons in a league.
    Color by season.
    Title includes per-league average Spearman r,p (from CSV).
    """
    df_l = pct_df[pct_df["league"] == league_key].copy()
    if df_l.empty:
        print(f"[WARN] No rows for league={league_key}")
        return

    title_league = LEAGUE_NAME_MAP.get(league_key, league_key.title())

    # per-league average Spearman
    row = sp_league.loc[sp_league["league"] == league_key]
    r_avg = float(row["spearman_r_avg"].iloc[0]) if not row.empty and "spearman_r_avg" in row.columns else np.nan
    p_avg = float(row["spearman_p_avg"].iloc[0]) if not row.empty and "spearman_p_avg" in row.columns else np.nan

    # prepare data
    df_l = df_l.dropna(subset=["first_half_win_pct_avg", "second_half_win_pct_avg"])
    # convert to percent 
    x = df_l["first_half_win_pct_avg"].values
    y = df_l["second_half_win_pct_avg"].values
    seasons = df_l["season"].astype(int).values

    # plot
    fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
    sc = ax.scatter(x, y, c=seasons, s=25, alpha=0.8)  # color by season

    # x axis formatting as percentages 
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

    # y = x reference line
    lim_min = 0.0
    lim_max = 1.0
    ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--")

    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_xlabel("First-Half Win % (avg across 10 seeds)")
    ax.set_ylabel("Second-Half Win % (avg across 10 seeds)")

    # title with per-league average Spearman (across seasons)
    title = f"{title_league}: First vs Second Half Win%\nSpearman r(avg) = {r_avg:.3f}, p(avg) = {p_avg:.3g}"
    ax.set_title(title)

    out_png = FIG_DIR / f"{league_key}_first_second_win_pct_avg.png"
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(f"✅ Saved {out_png}")

In [4]:
for k in ["bundesliga", "premier_league", "serie_a", "la_liga"]:
    plot_first_vs_second_avg_for_league(k)

✅ Saved /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/figures/bundesliga_first_second_win_pct_avg.png
✅ Saved /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/figures/premier_league_first_second_win_pct_avg.png
✅ Saved /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/figures/serie_a_first_second_win_pct_avg.png
✅ Saved /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/figures/la_liga_first_second_win_pct_avg.png
